In [ ]:
import argparse
import json
import subprocess
import sys
import time

import numpy as np
from scipy.optimize import brentq, fsolve

try:
    import h5py
except ImportError:
    h5py = None
    print("h5py not found. Everything works except sweep() file output.")
    print("Install with:  !pip install h5py")


# =====================================================================
# HAMILTONIAN
# =====================================================================


# F_0 = 1, F_1 = 1, F_2 = 2, ...
FIBONACCI = [1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377,
             610, 987, 1597, 2584, 4181, 6765]


def approximant(n):
    if n < 2 or n >= len(FIBONACCI):
        raise ValueError("n must satisfy 2 <= n < %d" % len(FIBONACCI))
    return FIBONACCI[n - 1] / FIBONACCI[n], FIBONACCI[n]


def phi_window(L):
    return 2.0 * np.pi / L


def phi_samples(L, n):
    return np.linspace(0.0, phi_window(L), n, endpoint=False)


def aa_potential(L, beta, lam, phi):
    i = np.arange(L)
    return lam * np.cos(2.0 * np.pi * beta * i + phi)


def build_normal(L, t, V, mu, U, n, A=0.0, hartree=True):
    h = np.zeros((L, L), dtype=complex)
    i = np.arange(L)
    j = (i + 1) % L
    h[i, j] = -t * np.exp(1j * A)
    h[j, i] = -t * np.exp(-1j * A)
    h[i, i] = V - mu - (0.5 * U * n if hartree else 0.0)
    return h


def build_bdg(L, t, V, mu, U, n, Delta, A=0.0, hartree=True):
    h = build_normal(L, t, V, mu, U, n, A, hartree)
    D = np.diag(np.asarray(Delta, dtype=complex))
    return np.block([[h, D], [D.conj().T, -h.conj()]])


def diagonalise(H):
    asym = np.max(np.abs(H - H.conj().T))
    if asym > 1e-10:
        raise ValueError("BdG matrix is not Hermitian (max asym %.3e)" % asym)
    E, W = np.linalg.eigh(H)
    return E, W


def fermi(E, T):
    if T <= 0.0:
        return np.where(E < 0.0, 1.0, np.where(E > 0.0, 0.0, 0.5))
    x = np.clip(E / T, -600.0, 600.0)
    return 1.0 / (np.exp(x) + 1.0)


def nambu_correlator(E, W, T):
    f = fermi(E, T)
    return (W.conj() * f[None, :]) @ W.T


def fields_from_correlator(R, U, L):
    idx = np.arange(L)
    Delta = -U * R[L + idx, idx]
    n = np.real(R[idx, idx]) + 1.0 - np.real(R[L + idx, L + idx])
    return Delta, n


def grand_potential(E, V, mu, U, n, Delta, T, L, hartree=True):
    Ep = E[L:]
    if T <= 0.0:
        band = -np.sum(Ep)
    else:
        x = Ep / (2.0 * T)
        ax = np.abs(x)
        band = -2.0 * T * np.sum(ax + np.log1p(np.exp(-2.0 * ax)))
    tr_h = np.sum(V - mu - (0.5 * U * n if hartree else 0.0))
    if U == 0.0:
        E0 = 0.0
    else:
        E0 = np.sum(np.abs(Delta) ** 2 / U)
        if hartree:
            E0 += np.sum(0.25 * U * n ** 2)
    return float(np.real(band + tr_h + E0))


def check_phs(E, W, L, tol=1e-8):
    return float(np.max(np.abs(E + E[::-1])))


# =====================================================================
# SELF-CONSISTENCY
# =====================================================================


class SCResult:

    def __init__(self, **kw):
        self.__dict__.update(kw)

    def __repr__(self):
        return ("SCResult(L=%d, converged=%s, iters=%d, |Delta|avg=%.6f, "
                "n_avg=%.6f, Omega=%.8f)"
                % (self.L, self.converged, self.iters,
                   np.mean(np.abs(self.Delta)), np.mean(self.n), self.Omega))


def _pack(Delta, n):
    return np.concatenate([Delta.real, Delta.imag, n])


def _unpack(x, L):
    return x[:L] + 1j * x[L:2 * L], x[2 * L:]


def _align_gauge(Delta_out, Delta_in, floor=1e-11):
    nrm = np.linalg.norm(Delta_out) * np.linalg.norm(Delta_in)
    if nrm < floor:
        return Delta_out
    ov = np.vdot(Delta_in, Delta_out)
    if np.abs(ov) < 1e-8 * nrm:
        return Delta_out
    return Delta_out * np.exp(-1j * np.angle(ov))


def _sc_map(x, L, t, V, mu, U, T, A, hartree):
    Delta_in, n_in = _unpack(x, L)
    H = build_bdg(L, t, V, mu, U, n_in, Delta_in, A, hartree)
    E, W = diagonalise(H)
    R = nambu_correlator(E, W, T)
    Delta_out, n_out = fields_from_correlator(R, U, L)
    Delta_out = _align_gauge(Delta_out, Delta_in)
    Om = grand_potential(E, V, mu, U, n_in, Delta_in, T, L, hartree)
    return _pack(Delta_out, n_out), Om, E, W, R


def solve(L, t, V, mu, U, T, A=0.0, Delta0=None, n0=None, hartree=True,
          tol=1e-10, max_iter=3000, mix=1.0, memory=6, warmup=15,
          warmup_mix=0.3, compare_normal=False, verbose=False):
    if Delta0 is None:
        Delta0 = np.full(L, 0.1 * t, dtype=complex)
    if n0 is None:
        n0 = np.ones(L)

    x = _pack(np.asarray(Delta0, dtype=complex), np.asarray(n0, dtype=float))

    xs, rs = [], []
    conv, it = False, 0
    Om = np.nan
    E = W = R = None

    for it in range(1, max_iter + 1):
        gx, Om, E, W, R = _sc_map(x, L, t, V, mu, U, T, A, hartree)
        r = gx - x
        res = np.max(np.abs(r))
        if verbose and it % 20 == 0:
            print("    it %4d  res %.3e" % (it, res))
        if res < tol:
            conv = True
            x = gx
            break

        if memory > 0 and it > warmup:
            xs.append(x.copy())
            rs.append(r.copy())
            if len(xs) > memory:
                xs.pop(0)
                rs.pop(0)
            m = len(xs)
            if m >= 2:
                Rm = np.array(rs).T
                try:
                    B = Rm.T @ Rm
                    B = B + 1e-12 * np.trace(B) / m * np.eye(m)
                    ones = np.ones(m)
                    c = np.linalg.solve(B, ones)
                    c = c / np.sum(c)
                    xbar = np.array(xs).T @ c
                    rbar = Rm @ c
                    x = xbar + mix * rbar
                    continue
                except np.linalg.LinAlgError:
                    pass
        step = warmup_mix if it <= warmup else mix
        x = x + step * r

    Delta, n = _unpack(x, L)
    H = build_bdg(L, t, V, mu, U, n, Delta, A, hartree)
    E, W = diagonalise(H)
    R = nambu_correlator(E, W, T)
    Om = grand_potential(E, V, mu, U, n, Delta, T, L, hartree)

    out = SCResult(L=L, t=t, V=V, mu=mu, U=U, T=T, A=A, hartree=hartree,
                   Delta=Delta, n=n, E=E, W=W, R=R, Omega=Om,
                   converged=conv, iters=it, residual=res, branch="sc")

    if compare_normal:
        nrm = solve(L, t, V, mu, U, T, A=A,
                    Delta0=np.zeros(L, dtype=complex), n0=n0, hartree=hartree,
                    tol=tol, max_iter=max_iter, mix=mix, memory=memory,
                    warmup=0, compare_normal=False)
        nrm.branch = "normal"
        if nrm.converged and nrm.Omega < out.Omega - 1e-12:
            nrm.rejected_Omega = out.Omega
            return nrm
        out.rejected_Omega = nrm.Omega
    return out


def solve_robust(L, t, V, mu, U, T, A=0.0, seed=None, hartree=True,
                 extra_seeds=(), **kw):
    cands = []
    n0 = seed.n if seed is not None else None

    seeds = []
    if seed is not None and np.mean(np.abs(seed.Delta)) > 1e-3:
        seeds.append(np.asarray(seed.Delta, dtype=complex))
    seeds.append(np.full(L, 0.5 * abs(U) + 0.1 * t, dtype=complex))
    seeds.append(np.full(L, 0.05 * t, dtype=complex))
    seeds.extend(np.asarray(e, dtype=complex) for e in extra_seeds)
    seeds.append(np.zeros(L, dtype=complex))

    for k, D0 in enumerate(seeds):
        is_normal = np.allclose(D0, 0.0)
        r = solve(L, t, V, mu, U, T, A=A, Delta0=D0, n0=n0, hartree=hartree,
                  warmup=0 if is_normal else kw.pop("warmup", 15)
                  if k == 0 else 15,
                  compare_normal=False, **kw)
        if r.converged:
            r.branch = "normal" if np.mean(np.abs(r.Delta)) < 1e-8 else "sc"
            cands.append(r)

    if not cands:
        raise RuntimeError("solve_robust: no seed converged at mu=%.4f" % mu)

    cands.sort(key=lambda r: r.Omega)
    best = cands[0]
    best.Omega_gap = (cands[1].Omega - best.Omega) if len(cands) > 1 else np.inf
    best.n_seeds_tried = len(seeds)
    return best


def solve_at_filling(L, t, V, U, T, n_target, A=0.0, mu_bracket=(-12.0, 12.0),
                     mu_tol=1e-9, seed=None, **kw):
    cache = {}

    def filling(mu):
        d0 = n0 = None
        if seed is not None:
            n0 = seed.n
            if np.mean(np.abs(seed.Delta)) >= 1e-3:
                d0 = seed.Delta
        r = solve(L, t, V, mu, U, T, A=A, Delta0=d0, n0=n0, **kw)
        cache[mu] = r
        return np.mean(r.n) - n_target

    lo, hi = mu_bracket
    flo, fhi = filling(lo), filling(hi)
    if flo > 0 or fhi < 0:
        raise ValueError("filling %.3f not bracketed by mu in [%.2f, %.2f] "
                         "(n(lo)=%.4f, n(hi)=%.4f)"
                         % (n_target, lo, hi, flo + n_target, fhi + n_target))

    while hi - lo > mu_tol:
        mid = 0.5 * (lo + hi)
        fm = filling(mid)
        if fm < 0:
            lo, flo = mid, fm
        else:
            hi, fhi = mid, fm

    mu = 0.5 * (lo + hi)
    if mu not in cache:
        filling(mu)
    return cache[mu]


# =====================================================================
# OBSERVABLES
# =====================================================================


def expect_onebody(R, O, L):
    Ruu = R[:L, :L]
    Rdd = R[L:, L:]
    val = np.sum(O * Ruu) + np.trace(O) - np.sum(O * Rdd.T)
    return float(np.real(val))


def hop_matrix(L, t, A=0.0):
    K = np.zeros((L, L), dtype=complex)
    i = np.arange(L)
    j = (i + 1) % L
    K[i, j] = -t * np.exp(1j * A)
    K[j, i] = -t * np.exp(-1j * A)
    return K


def current_matrix(L, t, A=0.0):
    J = np.zeros((L, L), dtype=complex)
    i = np.arange(L)
    j = (i + 1) % L
    J[i, j] = 1j * t * np.exp(1j * A)
    J[j, i] = -1j * t * np.exp(-1j * A)
    return J


def kinetic_energy(res):
    K = hop_matrix(res.L, res.t, res.A)
    return expect_onebody(res.R, K, res.L)


def diamagnetic_weight(res):
    return -kinetic_energy(res) / res.L


def stiffness_twist(L, t, V, mu, U, T, dA=0.01, richardson=True,
                    seed=None, tol=1e-9, **kw):
    worst = [0.0]

    def om(A, sd):
        r = solve(L, t, V, mu, U, T, A=A, tol=tol,
                  Delta0=None if sd is None else sd.Delta,
                  n0=None if sd is None else sd.n, **kw)
        worst[0] = max(worst[0], r.residual)
        return r.Omega, r

    O0, r0 = om(0.0, seed)

    def second_diff(h):
        Op, _ = om(h, r0)
        Om_, _ = om(-h, r0)
        return (Op - 2.0 * O0 + Om_) / (h * h) / L

    Dc = second_diff(dA)
    out = {"D_coarse": Dc, "Omega_0": O0, "res_0": r0}
    if richardson:
        Df = second_diff(0.5 * dA)
        out["D_fine"] = Df
        out["D_s"] = (4.0 * Df - Dc) / 3.0
    else:
        out["D_fine"] = np.nan
        out["D_s"] = Dc
    out["max_res"] = worst[0]
    out["D_s_conv_err"] = 4.0 * worst[0] ** 2 / (L * dA * dA)
    out["ok"] = bool(worst[0] < 1e-6)
    return out


def cdw_order(res):
    if res.L % 2 != 0:
        return float("nan")
    stag = (-1.0) ** np.arange(res.L)
    return float(0.5 * res.U * np.mean((res.n - 1.0) * stag))


def ipr(W, L, q=2):
    u = W[:L, :]
    v = W[L:, :]
    w = np.abs(u) ** 2 + np.abs(v) ** 2
    return np.sum(w ** q, axis=0)


def d2_energy_resolved(results, edges):
    Ls = np.array(sorted({r["L"] for r in results}), dtype=float)
    if len(Ls) < 3:
        raise ValueError("need at least 3 system sizes, ideally 5+")

    nb = len(edges) - 1
    centers = 0.5 * (edges[:-1] + edges[1:])
    D2 = np.full(nb, np.nan)
    D2e = np.full(nb, np.nan)
    r2 = np.full(nb, np.nan)

    for b in range(nb):
        lo, hi = edges[b], edges[b + 1]
        xs, ys = [], []
        for Lv in Ls:
            P2s = []
            for r in results:
                if r["L"] != Lv:
                    continue
                aE = np.abs(r["E"])
                m = (aE >= lo) & (aE < hi)
                if np.any(m):
                    P2s.append(r["P2"][m])
            if not P2s:
                break
            P2s = np.concatenate(P2s)
            if P2s.size < 5:
                continue
            xs.append(np.log(Lv))
            ys.append(np.mean(np.log(P2s)))
        if len(xs) < 3:
            continue
        xs, ys = np.array(xs), np.array(ys)
        A = np.vstack([xs, np.ones_like(xs)]).T
        coef, resid, _, _ = np.linalg.lstsq(A, ys, rcond=None)
        slope = coef[0]
        D2[b] = -slope
        yhat = A @ coef
        ss_res = np.sum((ys - yhat) ** 2)
        ss_tot = np.sum((ys - np.mean(ys)) ** 2)
        r2[b] = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
        dof = len(xs) - 2
        if dof > 0 and ss_tot > 0:
            s2 = ss_res / dof
            cov = s2 * np.linalg.inv(A.T @ A)
            D2e[b] = np.sqrt(cov[0, 0])
    return centers, D2, D2e, r2


def qfi_zero_T(res, O):
    if res.T > 0.0:
        raise ValueError("qfi_zero_T is valid only at T = 0. At finite T, "
                         "4*Var is an upper bound on F_Q, not F_Q.")
    L = res.L
    O = np.asarray(O, dtype=complex)
    GN = np.block([[O, np.zeros((L, L), dtype=complex)],
                   [np.zeros((L, L), dtype=complex), -O.T]])
    Ga = res.W.conj().T @ GN @ res.W
    occ = fermi(res.E, 0.0)
    wgt = np.outer(occ, 1.0 - occ)
    var = float(np.real(np.sum(np.abs(Ga) ** 2 * wgt)))
    return 4.0 * var


def qfi_current_density(res):
    J = current_matrix(res.L, res.t, res.A)
    return qfi_zero_T(res, J) / res.L


def pair_correlator(res):
    L = res.L
    R = res.R
    i = np.arange(L)
    rmax = L // 2
    out = np.zeros(rmax + 1)
    for r in range(rmax + 1):
        j = (i + r) % L
        F = R[L + j, i]
        out[r] = np.mean(np.abs(F))
    return np.arange(rmax + 1), out


def coherence_length(res, rmin=2, rmax=None):
    r, F = pair_correlator(res)
    if rmax is None:
        rmax = min(len(r) - 1, max(rmin + 5, len(r) // 2))
    m = (r >= rmin) & (r <= rmax) & (F > 1e-14)
    if np.count_nonzero(m) < 4:
        return np.nan, np.nan
    x = r[m].astype(float)
    y = np.log(F[m])
    A = np.vstack([x, np.ones_like(x)]).T
    coef, _, _, _ = np.linalg.lstsq(A, y, rcond=None)
    slope = coef[0]
    yhat = A @ coef
    ss_res = np.sum((y - yhat) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    xi = -1.0 / slope if slope < 0 else np.inf
    return float(xi), float(r2)


# =====================================================================
# SWEEP DRIVER
# =====================================================================


def git_hash():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception:
        return "unknown"


def _seed_fields(seed, floor=1e-3):
    if seed is None:
        return None, None
    n0 = seed.n
    D0 = seed.Delta
    if np.mean(np.abs(D0)) < floor:
        D0 = None
    return D0, n0


def run_point(L, beta, t, lam, phi, U, T, filling, mu, want_stiffness=True,
              seed=None, dA=0.01, hartree=True):
    V = aa_potential(L, beta, lam, phi)
    D0, n0 = _seed_fields(seed)

    if filling is not None:
        r = solve_at_filling(L, t, V, U, T, filling, seed=seed, tol=1e-10,
                             hartree=hartree, compare_normal=True)
        mu_used = r.mu
    else:
        r = solve_robust(L, t, V, mu, U, T, seed=seed, hartree=hartree,
                         tol=1e-10)
        mu_used = mu

    rec = {
        "lam": lam, "phi": phi, "mu": mu_used,
        "Delta_avg": float(np.mean(np.abs(r.Delta))),
        "Delta_std": float(np.std(np.abs(r.Delta))),
        "n_avg": float(np.mean(r.n)),
        "Omega": r.Omega,
        "converged": bool(r.converged),
        "iters": int(r.iters),
        "branch_sc": bool(getattr(r, "branch", "sc") == "sc"),
        "Omega_gap": float(getattr(r, "Omega_gap", np.nan)),
        "D_dia": diamagnetic_weight(r),
        "W_cdw": cdw_order(r),
    }
    xi, xi_r2 = coherence_length(r)
    rec["xi"] = xi
    rec["xi_r2"] = xi_r2
    if T <= 0.0:
        rec["qfi_J_density"] = qfi_current_density(r)
    else:
        rec["qfi_J_density"] = np.nan

    if want_stiffness:
        out = stiffness_twist(L, t, V, mu_used, U, T, dA=dA, seed=r,
                              hartree=hartree)
        rec["D_s"] = out["D_s"]
        rec["D_s_coarse"] = out["D_coarse"]
        rec["D_s_fine"] = out["D_fine"]
        rec["D_s_maxres"] = out["max_res"]
        rec["D_s_converr"] = out["D_s_conv_err"]
        rec["D_s_ok"] = bool(out["ok"])
        rec["D_s_steperr"] = abs(out["D_coarse"] - out["D_fine"])
    else:
        for k in ("D_s", "D_s_coarse", "D_s_fine", "D_s_steperr",
                  "D_s_maxres", "D_s_converr"):
            rec[k] = np.nan
        rec["D_s_ok"] = False

    fields = {
        "Delta_re": r.Delta.real, "Delta_im": r.Delta.imag,
        "n": r.n,
        "E_pos": r.E[L:],
        "P2_pos": ipr(r.W, L)[L:],
    }
    return rec, fields, r


def sweep(out_path, n_approx=12, t=1.0, lam_grid=(0.0, 4.0, 21), U=2.0, T=0.0,
          filling=0.9, mu=None, phis=16, stiffness=True, dA=0.01,
          both_directions=True, hartree=True, verbose=True):
    if h5py is None:
        raise ImportError("h5py required: pip install h5py")

    beta, L = approximant(n_approx)
    lams = np.linspace(lam_grid[0], lam_grid[1], int(lam_grid[2]))
    phi_list = phi_samples(L, int(phis))

    params = dict(n_approx=n_approx, beta=beta, L=L, t=t, U=U, T=T,
                  filling=filling, mu=mu, dA=dA, n_phi=int(phis),
                  hartree=bool(hartree),
                  lam_min=lam_grid[0], lam_max=lam_grid[1],
                  n_lam=int(lam_grid[2]), stiffness=bool(stiffness),
                  both_directions=bool(both_directions),
                  git=git_hash(), numpy=np.__version__,
                  timestamp=time.strftime("%Y-%m-%dT%H:%M:%S"))

    directions = ["fwd", "bwd"] if both_directions else ["fwd"]
    t0 = time.time()

    with h5py.File(out_path, "w") as f:
        for k, v in params.items():
            if isinstance(v, (bool, int, float, str, np.integer, np.floating)):
                f.attrs[k] = v
            else:
                f.attrs[k] = json.dumps(v)

        for direction in directions:
            order = np.arange(len(lams))
            if direction == "bwd":
                order = order[::-1]
            g = f.create_group(direction)

            for pi, phi in enumerate(phi_list):
                seed = None
                recs = []
                for oi in order:
                    lam = float(lams[oi])
                    rec, fields, r = run_point(
                        L, beta, t, lam, float(phi), U, T, filling, mu,
                        want_stiffness=stiffness, seed=seed, dA=dA,
                        hartree=hartree)
                    seed = r
                    recs.append((oi, rec, fields))
                    if verbose:
                        print("  %s phi=%2d lam=%.3f  |D|=%.5f  D_s=%.5f  "
                              "n=%.4f  (%.1fs)"
                              % (direction, pi, lam, rec["Delta_avg"],
                                 rec["D_s"], rec["n_avg"], time.time() - t0))

                gp = g.create_group("phi_%03d" % pi)
                gp.attrs["phi"] = float(phi)
                keys = [k for k in recs[0][1] if k not in ("phi", "lam")]
                idx = np.argsort([r[0] for r in recs])
                for k in keys:
                    arr = np.array([recs[i][1][k] for i in idx])
                    gp.create_dataset(k, data=arr)
                gp.create_dataset("lam", data=lams)
                for fk in recs[0][2]:
                    stack = np.array([recs[i][2][fk] for i in idx])
                    d = gp.create_dataset(fk, data=stack, compression="gzip")
                    d.attrs["axis0"] = "lam"
                    d.attrs["axis1"] = "site" if fk in ("Delta_re",
                                                        "Delta_im", "n") \
                        else "state"

    if verbose:
        print("wrote %s  (%.1f s)" % (out_path, time.time() - t0))
    return out_path


# =====================================================================
# VALIDATION
# =====================================================================


PASS, FAIL = "PASS", "FAIL"
_results = []


def _nonvacuous(r):
    return (np.mean(np.abs(r.Delta)) > 1e-2) and (0.05 < np.mean(r.n) < 1.95)


def _report(name, ok, detail):
    tag = PASS if ok else FAIL
    print("[%s] %-42s %s" % (tag, name, detail))
    _results.append(ok)
    return ok


def test_noninteracting_gap():
    L, t = 55, 1.0
    V = aa_potential(L, approximant(10)[0], 1.5, 0.3)
    r = solve(L, t, V, mu=-0.2, U=0.0, T=0.0, Delta0=np.full(L, 0.3 + 0j))
    m = np.max(np.abs(r.Delta))
    return _report("U=0 gives Delta=0", m < 1e-12, "max|Delta| = %.2e" % m)


def test_phs():
    L, t = 89, 1.0
    V = aa_potential(L, approximant(11)[0], 2.5, 1.1)
    n = 1.0 + 0.2 * np.cos(np.arange(L))
    D = 0.3 * np.exp(1j * np.linspace(0, 3, L))
    H = build_bdg(L, t, V, mu=-0.4, U=2.0, n=n, Delta=D, A=0.07)
    E, W = diagonalise(H)
    viol = check_phs(E, W, L)
    return _report("particle-hole symmetry of spectrum", viol < 1e-9,
                   "max|E_n + E_{-n}| = %.2e" % viol)


def _bcs_kspace(L, t, U, mu, T):
    k = 2.0 * np.pi * np.arange(L) / L

    def th(E):
        if T <= 0.0:
            return np.ones_like(E)
        return np.tanh(E / (2.0 * T))

    def eqs(x):
        D, n = x
        xi = -2.0 * t * np.cos(k) - (mu + 0.5 * U * n)
        E = np.sqrt(xi ** 2 + D ** 2)
        g = 1.0 - (U / L) * np.sum(th(E) / (2.0 * E))
        f = n - (1.0 / L) * np.sum(1.0 - (xi / E) * th(E))
        return [g, f]

    sol = fsolve(eqs, [0.5, 1.0], full_output=False)
    return sol[0], sol[1]


def test_clean_bcs_T0():
    L, t, U, mu = 89, 1.0, 2.0, -1.2
    V = np.zeros(L)
    r = solve(L, t, V, mu, U, T=0.0, tol=1e-11)
    Dk, nk = _bcs_kspace(L, t, U, mu, 0.0)
    dD = abs(np.mean(np.abs(r.Delta)) - abs(Dk))
    dn = abs(np.mean(r.n) - nk)
    ok = (dD < 1e-8) and (dn < 1e-8) and _nonvacuous(r)
    return _report("clean BCS T=0 vs analytic gap eq", ok,
                   "dDelta = %.2e, dn = %.2e (|D|=%.4f, n=%.4f)"
                   % (dD, dn, np.mean(np.abs(r.Delta)), np.mean(r.n)))


def test_clean_bcs_finiteT():
    L, t, U, mu, T = 89, 1.0, 2.0, -1.0, 0.15
    V = np.zeros(L)
    r = solve(L, t, V, mu, U, T=T, tol=1e-11)
    Dk, nk = _bcs_kspace(L, t, U, mu, T)
    dD = abs(np.mean(np.abs(r.Delta)) - abs(Dk))
    dn = abs(np.mean(r.n) - nk)
    ok = (dD < 1e-8) and (dn < 1e-8) and _nonvacuous(r)
    return _report("clean BCS T>0 vs analytic gap eq", ok,
                   "dDelta = %.2e, dn = %.2e (|D|=%.4f, n=%.4f)"
                   % (dD, dn, np.mean(np.abs(r.Delta)), np.mean(r.n)))


def test_stiffness_equals_diamagnetic():
    L, t, U = 56, 1.0, 3.0
    mu = -0.5 * U - 0.6
    V = np.zeros(L)
    out = stiffness_twist(L, t, V, mu, U, T=0.0, dA=0.01, tol=1e-11)
    Ds = out["D_s"]
    r0 = out["res_0"]
    Dd = diamagnetic_weight(r0)
    rel = abs(Ds - Dd) / abs(Dd)
    W = abs(cdw_order(r0))
    ok = (rel < 1e-5) and _nonvacuous(r0) and (W < 1e-6)
    return _report("clean BCS T=0: D_s == D_dia", ok,
                   "D_s = %.7f, D_dia = %.7f, rel = %.1e, n = %.4f, W = %.1e"
                   % (Ds, Dd, rel, np.mean(r0.n), W))


def test_aa_transition():
    t = 1.0
    ns = [8, 9, 10, 11, 12]           # L = 34, 55, 89, 144, 233
    out = {}
    for lam in (1.0, 2.0, 3.0):
        xs, ys = [], []
        for n in ns:
            beta, L = approximant(n)
            phis = phi_samples(L, 24)
            acc = []
            for phi in phis:
                V = aa_potential(L, beta, lam, phi)
                H = build_bdg(L, t, V, mu=0.0, U=0.0,
                              n=np.zeros(L), Delta=np.zeros(L))
                E, W = diagonalise(H)
                acc.append(np.log(ipr(W, L)[L:]))
            xs.append(np.log(L))
            ys.append(np.mean(np.concatenate(acc)))
        slope = np.polyfit(xs, ys, 1)[0]
        out[lam] = -slope
    ok = (out[1.0] > 0.85) and (out[3.0] < 0.15) and (0.15 < out[2.0] < 0.85)
    return _report("AA transition at lam = 2t (D2 pipeline)", ok,
                   "D2: lam=1 %.3f | lam=2 %.3f | lam=3 %.3f"
                   % (out[1.0], out[2.0], out[3.0]))


def test_filling_sum_rule():
    beta, L = approximant(10)
    V = aa_potential(L, beta, 1.2, 0.7)
    target = 0.85
    r = solve_at_filling(L, 1.0, V, U=2.0, T=0.05, n_target=target, tol=1e-10)
    d = abs(np.mean(r.n) - target)
    return _report("mu bisection hits target filling", d < 1e-6,
                   "|<n> - target| = %.2e at mu = %.6f" % (d, r.mu))


def test_qfi_against_bruteforce():
    L, t, U, mu = 21, 1.0, 2.0, -0.4
    V = aa_potential(L, approximant(8)[0] * 0 + 0.6180339887, 1.0, 0.2)
    r = solve(L, t, V, mu, U, T=0.0, tol=1e-11)
    J = current_matrix(L, t, 0.0)
    fq = qfi_zero_T(r, J)

    GN = np.block([[J, np.zeros((L, L), complex)],
                   [np.zeros((L, L), complex), -J.T]])
    Ga = r.W.conj().T @ GN @ r.W
    occ = (r.E < 0).astype(float)
    mean = np.sum(np.diag(Ga).real * occ)
    var = 0.0
    for m in range(2 * L):
        for n in range(2 * L):
            if occ[m] > 0 and occ[n] == 0:
                var += abs(Ga[m, n]) ** 2
    rel = abs(fq - 4 * var) / max(abs(4 * var), 1e-12)
    return _report("QFI Wick formula, vectorised vs loop", rel < 1e-10,
                   "F_Q = %.8f, 4Var = %.8f" % (fq, 4 * var))


def test_pseudospin_degeneracy():
    L, t, U = 34, 1.0, 3.0
    mu, V = -0.5 * U, np.zeros(L)
    stag = (-1.0) ** np.arange(L)
    inv, oms = [], []
    for w in (0.0, 0.3, 0.6, 0.8):
        D0 = np.full(L, np.sqrt(max(1 - w * w, 0.0)) * 0.93 + 0j)
        n0 = 1.0 + w * 0.35 * stag
        r = solve(L, t, V, mu, U, 0.0, Delta0=D0, n0=n0, tol=1e-12, warmup=0)
        W = cdw_order(r)
        inv.append(np.mean(np.abs(r.Delta)) ** 2 + W ** 2)
        oms.append(r.Omega)
    dinv = max(inv) - min(inv)
    dom = max(oms) - min(oms)
    ok = (dinv < 1e-8) and (dom < 1e-8)
    return _report("pseudospin SU(2): |D|^2 + W^2 invariant", ok,
                   "spread in |D|^2+W^2 = %.2e, in Omega = %.2e" % (dinv, dom))


def test_phi_window_is_distinct():
    beta, L = approximant(8)                      # L = 34

    def n_distinct(phis):
        seen = []
        for ph in phis:
            v = aa_potential(L, beta, 1.0, ph)
            dup = any(any(np.allclose(np.roll(v, -m), w, atol=1e-10)
                          for m in range(L)) for w in seen)
            if not dup:
                seen.append(v)
        return len(seen)

    naive = n_distinct(np.linspace(0, 2 * np.pi, L, endpoint=False))
    good = n_distinct(phi_samples(L, 12))
    ok = (naive == 1) and (good == 12)
    return _report("phi window: 2pi/L gives distinct samples", ok,
                   "naive [0,2pi) grid at N=L: %d distinct | phi_samples(12): "
                   "%d distinct" % (naive, good))


def test_sweep_direction_independence():
    beta, L = approximant(8)
    t, U, mu = 1.0, 3.0, -0.5 * 3.0 - 0.6
    lams = [0.0, 1.0, 2.0, 3.0]

    def branch(order):
        prev, got = None, {}
        for lam in order:
            V = aa_potential(L, beta, lam, 0.0)
            r = solve_robust(L, t, V, mu, U, 0.0, seed=prev, tol=1e-10)
            prev = r
            got[lam] = np.mean(np.abs(r.Delta))
        return got

    fwd = branch(lams)
    bwd = branch(lams[::-1])
    worst = max(abs(fwd[k] - bwd[k]) for k in lams)
    return _report("sweep direction independence (no fake boundary)",
                   worst < 1e-6,
                   "max |fwd - bwd| over lam = %.2e" % worst)


def run_validation():
    del _results[:]
    print("=" * 74)
    print("BdG validation cascade")
    print("=" * 74)
    test_noninteracting_gap()
    test_phs()
    test_clean_bcs_T0()
    test_clean_bcs_finiteT()
    test_stiffness_equals_diamagnetic()
    test_filling_sum_rule()
    test_qfi_against_bruteforce()
    test_aa_transition()
    test_pseudospin_degeneracy()
    test_phi_window_is_distinct()
    test_sweep_direction_independence()
    print("=" * 74)
    n_ok = sum(_results)
    print("%d / %d passed" % (n_ok, len(_results)))
    print("=" * 74)
    return 0 if n_ok == len(_results) else 1


# =====================================================================
# RUN THIS
# =====================================================================

run_validation()

BdG validation cascade
[PASS] U=0 gives Delta=0                          max|Delta| = 0.00e+00
[PASS] particle-hole symmetry of spectrum         max|E_n + E_{-n}| = 5.33e-15
[PASS] clean BCS T=0 vs analytic gap eq           dDelta = 3.00e-12, dn = 6.72e-14 (|D|=0.3443, n=0.9028)
[PASS] clean BCS T>0 vs analytic gap eq           dDelta = 1.34e-13, dn = 3.77e-15 (|D|=0.2548, n=1.0000)
[PASS] clean BCS T=0: D_s == D_dia                D_s = 0.7773809, D_dia = 0.7773809, rel = 6.0e-10, n = 0.5461, W = 5.9e-17
[PASS] mu bisection hits target filling           |<n> - target| = 3.20e-10 at mu = -1.144066
[PASS] QFI Wick formula, vectorised vs loop       F_Q = 92.48770245, 4Var = 92.48770245
[PASS] AA transition at lam = 2t (D2 pipeline)    D2: lam=1 0.967 | lam=2 0.543 | lam=3 0.001
[PASS] pseudospin SU(2): |D|^2 + W^2 invariant    spread in |D|^2+W^2 = 1.78e-12, in Omega = 1.42e-14
[PASS] phi window: 2pi/L gives distinct samples   naive [0,2pi) grid at N=L: 1 distinct | phi_samples(12): 12 d

0